# NIOS PDF Extraction — marker-pdf on Kaggle

**What this notebook does:**
1. Auto-discovers **all subjects** from the `nios-chapter-pdfs` dataset
2. Checks which chapters are already extracted (resumable — skips existing JSON files)
3. Tracks overall progress in `_progress.json`
4. Saves `Chapter N.json` per chapter → `/kaggle/working/extracted/class<level>/<subject>/chapters/`

**Setup:**
1. Add **`dipankaj/nios-chapter-pdfs`** as a dataset input
2. *(Optional)* Edit `SUBJECTS_TO_PROCESS` to target a single subject or leave as `"all"`
3. Enable GPU accelerator **(T4)** → **Run All** — it will only extract what isn't done yet

**Dataset layout:**
```
/kaggle/input/nios-chapter-pdfs/
  class10/<subject-id>/Chapter N.pdf
  class12/<subject-id>/Chapter N.pdf
```

**Output layout:**
```
/kaggle/working/extracted/
  _progress.json                       ← global progress tracker (all subjects)
  class10/<subject-id>/chapters/
    Chapter N.json
    _manifest.json
  class12/<subject-id>/chapters/
    Chapter N.json
    _manifest.json
```


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
from pathlib import Path
import json

# Which subjects to process:
#   "all"                        → every subject found in the dataset  (default)
#   "maths-12"                   → single subject slug
#   ["maths-12", "physics-12"]   → specific list
SUBJECTS_TO_PROCESS = "all"

_DATASET_ROOT = Path("/kaggle/input/nios-chapter-pdfs")
WORKING_DIR   = Path("/kaggle/working/extracted")
PROGRESS_FILE = WORKING_DIR / "_progress.json"

MARKER_FORMAT = "json"   # options: json, markdown, html
RESUME        = True     # skip chapters that are already extracted


In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q docling


In [ ]:
# ── Discover subjects & build work plan ───────────────────────────────────────
import json, re as _re
from pathlib import Path

# ── Load or init progress tracker ────────────────────────────────────────────
if PROGRESS_FILE.exists():
    progress = json.loads(PROGRESS_FILE.read_text())
    print(f"📂 Progress loaded — {len(progress['subjects'])} subject(s) tracked so far")
else:
    progress = {"last_updated": None, "subjects": {}}
    print("📂 No existing progress — starting fresh")

# ── Discover all subjects in the dataset ─────────────────────────────────────
def _discover_subjects():
    dirs = []
    for cls_dir in sorted(_DATASET_ROOT.glob("class*")):
        if cls_dir.is_dir():
            for subj_dir in sorted(cls_dir.iterdir()):
                if subj_dir.is_dir() and list(subj_dir.glob("*.pdf")):
                    dirs.append(subj_dir)
    return dirs

all_subject_dirs = _discover_subjects()

if SUBJECTS_TO_PROCESS == "all":
    subject_dirs = all_subject_dirs
elif isinstance(SUBJECTS_TO_PROCESS, str):
    subject_dirs = [d for d in all_subject_dirs if d.name == SUBJECTS_TO_PROCESS]
else:
    subject_dirs = [d for d in all_subject_dirs if d.name in SUBJECTS_TO_PROCESS]

def _sort_key(p):
    m = _re.search(r"\d+", p.stem)
    return (0 if p.stem.startswith("Chapter") else 1, int(m.group()) if m else 999)

# ── Build per-subject work plan ───────────────────────────────────────────────
print(f"\nDataset : {_DATASET_ROOT}  ({len(all_subject_dirs)} subjects found)\n")
print(f"{'Subject':<22}  {'PDFs':>4}  {'Done':>4}  {'Left':>4}  Status")
print("─" * 52)

subjects_info = []
for subj_dir in subject_dirs:
    subject_id  = subj_dir.name
    class_level = subject_id.rsplit("-", 1)[-1]
    out_dir     = WORKING_DIR / f"class{class_level}" / subject_id / "chapters"
    pdfs        = sorted(subj_dir.glob("*.pdf"), key=_sort_key)

    meta_path = subj_dir / "_manifest.json"
    subject_name = (
        json.loads(meta_path.read_text()).get("subject_name", subject_id)
        if meta_path.exists() else subject_id
    )

    done = set()
    if RESUME and out_dir.exists():
        done = {
            jf.stem for jf in out_dir.glob("*.json")
            if jf.name != "_manifest.json" and jf.stat().st_size > 100
        }

    pending = [p for p in pdfs if p.stem not in done]
    subjects_info.append({
        "id": subject_id, "name": subject_name, "class_level": class_level,
        "dir": subj_dir, "out_dir": out_dir,
        "pdfs": pdfs, "done": done, "pending": pending,
    })

    tag = "✅ complete" if not pending else ("⏳ partial" if done else "🆕 new")
    print(f"  {subject_id:<22}  {len(pdfs):>4}  {len(done):>4}  {len(pending):>4}  {tag}")

total_pending = sum(len(s["pending"]) for s in subjects_info)
print(f"\n{'Total chapters to extract:':<30} {total_pending}")


In [ ]:
# ── Extract with Docling ──────────────────────────────────────────────────────
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
import traceback, time

pipeline_options = PdfPipelineOptions()
pipeline_options.images_scale = 1.0
pipeline_options.generate_page_images = False
pipeline_options.generate_picture_images = True

doc_converter = DocumentConverter(
    allowed_formats=[InputFormat.PDF],
    format_options={InputFormat.PDF: pipeline_options}
)

def process_pdf(pdf_path: Path, output_dir: Path):
    try:
        conv_res = doc_converter.convert(pdf_path)
        doc = conv_res.document
        
        # Save JSON representation
        json_path = output_dir / f"{pdf_path.stem}.json"
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(doc.export_to_dict(), f, indent=2)
            
        # Save embedded pictures/figures
        image_count = 0
        # Docling stores extracted pictures in the document's elements
        for element, _ in doc.iterate_items():
            if hasattr(element, "image") and element.image is not None:
                image_count += 1
                img_filename = f"{pdf_path.stem}_image_{image_count}.png"
                element.image.save(output_dir / img_filename)
                # Add a ref to the JSON item so we know what file it is
                element.image_filename = img_filename 
        
        # Re-save JSON to include the new image_filename attributes we just bound
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(doc.export_to_dict(), f, indent=2)
            
        return True
    except Exception as e:
        print(f"    ❌ Error: {e}")
        traceback.print_exc()
        return False

print("Starting Docling extraction...\n")

for si, subj in enumerate(subjects_info, 1):
    subject_id = subj["id"]
    chapters = subj["pending"]
    
    if not chapters:
        print(f"[{si}/{len(subjects_info)}] ⏭️  {subj[id]} — already complete")
        continue
    
    print(f"\n[{si}/{len(subjects_info)}] 📚 {subject_id}\n" + "="*40)
    
    subject_out_dir = WORKING_DIR / f"class{subj[class_level]}" / subject_id / "chapters"
    subject_out_dir.mkdir(parents=True, exist_ok=True)
    
    success_count = 0
    for idx, pdf_path in enumerate(chapters, 1):
        json_path = subject_out_dir / f"{pdf_path.stem}.json"
        if json_path.exists():
            print(f"  [{idx:2d}/{len(chapters)}] ⏭️  Skipping {pdf_path.stem} (already done)")
            success_count += 1
            continue
            
        print(f"  [{idx:2d}/{len(chapters)}] ⚙️  Extracting {pdf_path.stem}...")
        ok = process_pdf(pdf_path, subject_out_dir)
        if ok:
            print(f"         ✅ Saved JSON + Images")
            success_count += 1
            
    # Update progress record
    progress["subjects"][subject_id] = {
        "status": "completed" if success_count == len(chapters) else "partial",
        "chapters_done": success_count,
        "total_chapters": len(chapters),
        "completed_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    }
    progress["last_updated"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    # Save checkpoint after each subject
    with open(PROGRESS_FILE, "w") as f:
        json.dump(progress, f, indent=2)
    print(f"  💾 Progress saved")

print("\n\n🎉 Extraction finished.")


In [ ]:
# ── Write manifests & update progress tracker ─────────────────────────────────
import datetime

now = datetime.datetime.now(datetime.timezone.utc).isoformat()

for subj in subjects_info:
    if not subj["out_dir"].exists():
        continue

    json_files = sorted(
        [f for f in subj["out_dir"].glob("*.json") if f.name != "_manifest.json"],
        key=_sort_key,
    )

    manifest = {
        "subject":       subj["id"],
        "subject_name":  subj["name"],
        "class_level":   subj["class_level"],
        "extracted_at":  now,
        "marker_format": MARKER_FORMAT,
        "chapters": [
            {"name": f.stem, "file": f.name, "size_bytes": f.stat().st_size}
            for f in json_files
        ],
    }
    with open(subj["out_dir"] / "_manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)

    stats  = subj.get("stats", {})
    n_ok   = len(json_files)
    n_fail = stats.get("fail", 0)
    progress["subjects"][subj["id"]] = {
        "class_level":     subj["class_level"],
        "total_pdfs":      len(subj["pdfs"]),
        "extracted":       n_ok,
        "failed":          n_fail,
        "failed_chapters": stats.get("failed", []),
        "status":          "complete" if n_ok == len(subj["pdfs"]) and n_fail == 0 else "partial",
        "last_run":        now,
    }

progress["last_updated"] = now
with open(PROGRESS_FILE, "w") as f:
    json.dump(progress, f, indent=2)

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"{'Subject':<22}  {'Total':>5}  {'Done':>5}  {'Fail':>5}  Status")
print("─" * 56)
for sid, info in progress["subjects"].items():
    icon = "✅" if info["status"] == "complete" else "⚠️ "
    print(f"  {sid:<22}  {info['total_pdfs']:>5}  {info['extracted']:>5}  {info['failed']:>5}  {icon}")
print(f"\n📋 Progress saved → {PROGRESS_FILE}")


In [ ]:
# ── Preview one chapter from any extracted subject ────────────────────────────
import re
_all_chapter_files = []
for _d in sorted(WORKING_DIR.glob("class*/*/chapters")):
    _all_chapter_files.extend(sorted(
        _d.glob("Chapter *.json"),
        key=lambda p: int(m.group()) if (m := re.search(r"\d+", p.stem)) else 999,
    ))

if _all_chapter_files:
    sample = _all_chapter_files[0]
    with open(sample) as f:
        data = json.load(f)

    rel = sample.relative_to(WORKING_DIR)
    print(f"Preview : {rel}")
    print(f"Docling Nodes: {len(data.get(texts, []))} texts, {len(data.get(pictures, []))} pictures")
else:
    print("No extracted chapters found yet.")
